# Susceptibility, diffusion, T2, and T2\*

This notebook builds on the previous workshop sections by adding a
susceptibility-induced off-resonance field. Spins diffuse through a
geometry, accumulate phase in the local field, and contribute to the
measured signal. We will compare gradient-echo and spin-echo signals,
then inspect the phases that explain their different sensitivity to
susceptibility.

## 1. Define a susceptibility geometry

We model spherical susceptibility inclusions in a periodic volume. The
surrounding water is the signal of interest; the spheres are assigned a
susceptibility of `1 ppm`.

In [1]:
using Random
using MCMRSimulator
using MRIBuilder
using CairoMakie

Random.seed!(1)
set_theme!(Theme(fontsize=20))

repeat_size = 10.0  # μm
positions, radii = random_positions_radii(
    (repeat_size, repeat_size, repeat_size),
    0.3,
    3;
    mean=1.0,
    variance=0.0,
)

function sphere_geometry(radius_scale)
    return Spheres(
        radius=radii .* radius_scale,
        position=positions .* radius_scale,
        repeats=(repeat_size * radius_scale,
                 repeat_size * radius_scale,
                 repeat_size * radius_scale),
        susceptibility=1.0,
    )
end

geometry = sphere_geometry(1.0)
plot(geometry)

The sphere radii are all `1 μm`; their positions are randomly generated
at a volume fraction of `0.3`. The geometry repeats every `10 μm` in
each direction. The surrounding space represents water and will be
selected with `Subset(inside=false)`.

[`random_positions_radii`](https://pages.fmrib.ox.ac.uk/ndcn0236/mcmrsimulator.jl/dev/api/#MCMRSimulator.Geometries.User.RandomDistribution.random_positions_radii)
generates non-overlapping sphere positions and radii in a box, aiming
for the requested volume fraction. Here, `3` selects three-dimensional
spheres, while `mean=1.0` and `variance=0.0` give every sphere the same
radius. It is designed to be used as a repeating geometry, so the box
size is set to the requested repeat size. The function returns the
positions and radii of the spheres, which are then used to create a
`Spheres` geometry. The `susceptibility` parameter sets the
susceptibility of the spheres to `1 ppm`.

We can also use `random_positions_radii` to generate a geometry for
`Cylinders` or `Annuli` by changing the `dimension` argument from `3` to
`2`.

To change the characteristic inset radius `R` while keeping the same
packing pattern, we scale the sampled positions, radii, and repeat
lengths together. This common rescaling preserves the relative spacing
and volume fraction of the inclusions. We reuse the original `positions`
and `radii` rather than calling `random_positions_radii` again, so each
radius value uses the same underlying random arrangement.

The susceptibility creates a spatially varying off-resonance field. Here
we plot that field in a plane perpendicular to x, overlaid with the
inclusion locations:

In [1]:
plot_plane = PlotPlane(:x, size=15)
field_figure, field_axis, field_plot = plot_off_resonance(
    plot_plane,
    geometry;
    ngrid=300,
    colormap=:coolwarm,
)
field_axis.aspect = DataAspect()
plot!(field_axis, plot_plane, geometry, color=:white, transparency=true)
Colorbar(field_figure[1, 2], field_plot; label="Off-resonance shift (ppm)")
field_figure

The colours indicate positive and negative off-resonance shifts. Moving
water spins sample different parts of this field and accumulate
different phases.

## 2. Compare gradient echo and spin echo

We use a gradient echo and a spin echo with the same `10 ms` echo time.
The gradient echo does not refocus phase dispersion from the
susceptibility field; the spin echo reverses the phase accumulated
before its refocusing pulse.

In [1]:
TE = 10.0  # ms
gradient_echo = MRIBuilder.GradientEcho(TE=TE)
spin_echo = MRIBuilder.SpinEcho(TE=TE)

MRIBuilder.plot_sequence(gradient_echo)
MRIBuilder.plot_sequence(spin_echo)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************


<img width=600 height=450 style='object-fit: contain; height: auto;' src="data:image/png;base64, iVBORw0KGgoAAAANSUhEUgAAAlgAAAHCCAYAAAAzc7dkAAAAAXNSR0IArs4c6QAAAARnQU1BAACxjwv8YQUAAAAgY0hSTQAAeiYAAICEAAD6AAAAgOgAAHUwAADqYAAAOpgAABdwnLpRPAAAAAlwSFlzAAAOxAAADsQBlSsOGwAAIABJREFUeAHtwQlwF+SB8OFfkjeLJCg3AaKQch/hXgQGakFAsYt2q5J4IIdQPIpSC6ywpYBVEKNTrUo8YEVOEcGWrnWZEVjKPSogQu2WAAZEhFYRHEFuvu8/s9nJMFzKq4nwe55w4v9DkiRJ0QQkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSLmKPP/44W7dupUiTJk247777kKTzEZCki9jrr7/O6tWrKXLNNddw3333IUnnIyBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiqgb8Xq1atZvnw5xQ0dOpSkpCQkSdKFLaBvxVtvvcXo0aMp7oEHHiAlJQVJknRhC0iSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKarA98Svf/1rPv/8c06WlJRERkYGDRs25Cc/+Qn/9E//RHGLFi3i97//PWczcOBAWrZsiSRJ0vkKfE9MmTKFjz/+mDPJyspi/vz5NG/enCLr1q1j4sSJnE3nzp1p2bIlkiRJ5yvwPVKmTBl27dpFcUePHmXbtm088cQTzJ49m9tvv50NGzZwst/+9rf079+f00lPT0eSJCmGwPdMhQoVOFmVKlWYPn06q1evZuPGjWzfvp1atWpRXNmyZalQoQKSJEnftsAFIoRAkyZNKCwsZMeOHdSqVYuvq6CggHnz5tG6dWuuueYainv99dfZtGkT/fv3JyMjA0mSpNMJXEAKCwtJTk6mTp06fBM/+MEPmDdvHuPGjWPjxo3Url2bhHXr1pGbm8v111/PiBEjkCRJOpPABWD//v3k5eXxwQcfcM8991C9enVO9sEHH7BgwQJOpUWLFtSoUYMQAtOmTaN169YMGDCAhQsXcuTIEfr370+VKlV48cUXOR979+4lJSUFSaXH0aNHKe7o0aPs3bsXSaXTZZddRnJyMqVd4Hvk0KFDVK9eneKOHTvGZ599xokTJxg1ahRjx47lVJ555hmeeeYZTmX69On07t2bhMaNGzNu3DiGDh3Kiy++yCeffML69ev505/+RJUqVThXjRo14ic/+QmfffYZy5cvJ6FKlSpIKt0WL15MxYoVkVQ6vffee7Ro0YLSLvA9kpycTMuWLSnuxIkT7Nixgw8++IBZs2bRv39/6tSpw8n69+/P1Vdfzal07NiR4n7xi1/whz/8gaFDh3Lo0CHuvvtufvzjH/N19OrVi169erFz507mzJmDJEk6f5mZmXwfBL5HUlNTWbBgAacyefJkfvazn/Hv//7vzJ49m5NdeeWV9O7dm3ORnJzMCy+8QJMmTUhPT+fxxx/nm6pZsya/+MUvkCRJF4/ABWLgwIH827/9G++++y4x/Md//AcJ+/fv56WXXuL+++9HkiTpXAQuIBkZGfz973/nfP35z3/mySef5IEHHuDtt99m5MiR/PjHP6ZevXpIkiSdTeACcvz4cb788kvOxxdffEHfvn2pW7cu48aNY/v27bRs2ZJ+/fqxdOlSkpOTkSRJOpPABaR8+fIcPnyYgoIC6tevzzcxePBgPvroI5YuXUrZsmVp2LAhY8eOZcSIETz11FP88pe/RJIk6UwCF5B//ud/5p133mHIkCG8+eabfF3z5s1j+vTpDBkyhI4dO1Jk2LBhvPbaa4waNYqePXvSoEEDJEmSTifwPbFjxw7OJj8/n/z8fIobNmwYw4YN41zcdNNNnDhxgpOlpKTw7rvvIkmSdC4CkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZJUiu3fv5/bb7+d4gYOHEjPnj0prQKSJEml2OHDh5k/fz7FdenShdIsIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEm

Next we vary the inset radius `R` while keeping diffusivity and the
sequences fixed. This shows how the characteristic size of the
susceptibility sources changes phase dispersion. We read out only water
spins and record the per-spin transverse signal for both sequences. The
radius values are spaced logarithmically from `0.5` to `50 μm`.

In [1]:
inclusion_radii = 10.0 .^ range(log10(0.5), log10(50.0); length=9)
base_radius = first(radii)
diffusivity = 2.0  # μm²/ms, held fixed in this comparison
water = Subset(inside=false)
sequences = [gradient_echo, spin_echo]
transverse_signals = zeros(length(inclusion_radii), length(sequences))

for index in eachindex(inclusion_radii)
    simulation = Simulation(
        sequences;
        geometry=sphere_geometry(inclusion_radii[index] / base_radius),
        diffusivity=diffusivity,
        verbose=false,
    )
    signal = readout(simulation; subset=water, target_snr=5)
    transverse_signals[index, :] = transverse.(signal; mean=true)
end

gradient_transverse = transverse_signals[:, 1]
spin_transverse = transverse_signals[:, 2]

signal_figure = Figure(size=(750, 450))
signal_axis = Axis(
    signal_figure[1, 1];
    xlabel="Mean susceptibility-inclusion radius R (μm)",
    ylabel="Mean water transverse signal",
    xscale=log10,
)
lines!(signal_axis, inclusion_radii, gradient_transverse, linewidth=3, label="Gradient echo (T2*)")
lines!(signal_axis, inclusion_radii, spin_transverse, linewidth=3, label="Spin echo (T2)")
axislegend(signal_axis, position=:lb)
signal_figure

The gradient-echo signal retains dephasing from the local field at
readout. The spin echo refocuses static phase offsets, but diffusion
through the field means that the phase accumulated before and after the
refocusing pulse is not always identical.

## 3. Apparent R2 and R2\*

We can summarize the signal loss at this echo time as apparent rates,
`-log(S)/TE`. No explicit transverse relaxation was added to these
simulations, so these rates describe signal loss from susceptibility and
diffusion rather than intrinsic material relaxation rates.

In [1]:
apparent_r2star = -log.(max.(gradient_transverse, eps())) ./ TE
apparent_r2 = -log.(max.(spin_transverse, eps())) ./ TE

rate_figure = Figure(size=(750, 450))
rate_axis = Axis(
    rate_figure[1, 1];
    xlabel="Mean susceptibility-inclusion radius R (μm)",
    ylabel="Apparent relaxation rate (ms⁻¹)",
    xscale=log10,
)
lines!(rate_axis, inclusion_radii, apparent_r2star, linewidth=3, label="Gradient echo (R2*)")
lines!(rate_axis, inclusion_radii, apparent_r2, linewidth=3, label="Spin echo (R2)")
axislegend(rate_axis, position=:lt)
rate_figure

These are effective rates at a single echo time, not exponential fits
across multiple echo times. Susceptibility and diffusion can produce
non-exponential signal decay, and the spin echo only refocuses the
reversible part of the dephasing.

## 4. Inspect phase accumulation around the spin echo

Finally, we track water-spin phase at the middle and end of the echo for
small, medium, and large insets. The phase differences from the two
halves show how inset size changes refocusing when spins move through
the field.

In [1]:
phase_nspins = 2_000
spin_phases(snapshot) = [spin.orientations[1].phase for spin in snapshot]
phase_radii = [0.5, 5, 50.]
phase_snapshots = map(phase_radii) do radius
    simulation = Simulation(
        sequences;
        geometry=sphere_geometry(radius / base_radius),
        diffusivity=diffusivity,
        verbose=false,
    )
    readout(
        phase_nspins,
        simulation,
        [TE / 2, TE];
        return_snapshot=true,
        subset=water,
    )
end

phase_data = map(phase_snapshots) do snapshots
    initial_phase = -90.0
    phase_at_half = spin_phases(snapshots[1, 1]) .- initial_phase
    phase_at_echo = spin_phases(snapshots[1, 2]) .- initial_phase
    (
        first_half=phase_at_half,
        second_half=-(phase_at_echo .- phase_at_half),
    )
end

phase_limit = maximum(abs, reduce(vcat, [
    values
    for data in phase_data
    for values in (data.first_half, data.second_half)
]))
phase_limit = max(phase_limit, 1.0)

phase_figure = Figure(size=(1500, 500))
for (index, (radius, data)) in enumerate(zip(phase_radii, phase_data))
    axis = Axis(
        phase_figure[1, index];
        title="R = $(radius) μm",
        xlabel="First-half phase (degrees)",
        ylabel="Refocused second-half phase (degrees)",
        limits=(-phase_limit, phase_limit, -phase_limit, phase_limit),
    )
    scatter!(axis, data.first_half, data.second_half; markersize=2, alpha=0.25)
end
phase_figure

The distributions show how changing the inset size affects phase
accumulation for the same diffusivity. The gradient echo retains phase
dispersion at readout, while the spin echo reverses the second-half
contribution; diffusion can still make refocusing incomplete when spins
sample different fields in the two halves.